# Ensemble Prediction with NPClassifierEnsemble

This notebook shows how to use `NPClassifierEnsemble` to make **hierarchically consistent** predictions from three separately trained models.

The ensemble wraps the three hierarchical models (pathway / superclass / class) and applies the NP-Classifier voting algorithm:

1. Each model makes independent predictions.
2. Pathway is decided by **majority vote** across three sources (direct + class-derived + superclass-derived).
3. Superclass and class predictions are **filtered by the ontology** so only labels consistent with the voted pathway survive.

**Prerequisites:** Trained checkpoints from `01_training.ipynb` (or equivalent).

## 0. Imports

In [ ]:
import pandas as pd

from torch_np_classifier import (
    NPClassifierFeaturizer,
    NPClassifierLightning,
    NPClassifierEnsemble,
)

## 1. Load the three hierarchical models

In [ ]:
PATHWAY_CKPT = "checkpoints/hierarchical/pathway/pathway_np_classifier.ckpt"
SUPERCLASS_CKPT = "checkpoints/hierarchical/superclass/superclass_np_classifier.ckpt"
CLASS_CKPT = "checkpoints/hierarchical/class/class_np_classifier.ckpt"

pathway_model = NPClassifierLightning.load_from_checkpoint(PATHWAY_CKPT)
superclass_model = NPClassifierLightning.load_from_checkpoint(SUPERCLASS_CKPT)
class_model = NPClassifierLightning.load_from_checkpoint(CLASS_CKPT)

print("pathway model    — outputs:", pathway_model.model.fc_out.out_features)
print("superclass model — outputs:", superclass_model.model.fc_out.out_features)
print("class model      — outputs:", class_model.model.fc_out.out_features)

## 2. Build the ensemble

`NPClassifierEnsemble.from_label_names_file` reads the bundled `label_names.pkl` (730 ordered names) and splits them automatically into the three levels.

In [ ]:
ensemble = NPClassifierEnsemble.from_label_names_file(
    pathway_model,
    superclass_model,
    class_model,
    # Optional: tune voting thresholds
    pathway_threshold=0.5,
    superclass_threshold=0.3,
    class_threshold=0.1,
)

print(
    f"Pathway labels    ({len(ensemble._pathway_labels)}):    {ensemble._pathway_labels}"
)
print(
    f"Superclass labels ({len(ensemble._superclass_labels)}): first 5 = {ensemble._superclass_labels[:5]}"
)
print(
    f"Class labels      ({len(ensemble._class_labels)}): first 5 = {ensemble._class_labels[:5]}"
)

## 3. Predict a single SMILES

`ensemble.predict(smiles)` returns a dict with keys `pathway`, `superclass`, `class`, and `isglycoside`.

In [ ]:
# Quercetin — a well-known flavonoid
quercetin_smiles = "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12"

result = ensemble.predict(quercetin_smiles, check_glycoside=True)

print("Quercetin prediction")
print(f"  Pathway    : {result['pathway']}")
print(f"  Superclass : {result['superclass']}")
print(f"  Class      : {result['class']}")
print(f"  Glycoside  : {result['isglycoside']}")

## 4. Predict a batch of molecules

In [ ]:
MOLECULES = {
    "caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
    "quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "morphine": "CN1CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](O)C=C[C@@H]3[C@@H]1C5",
    "cholesterol": "[C@@H]1(CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CC=C4[C@@]3(CC[C@@H](C4)O)C)[C@H](CCCC(C)C)C)C",
    "sucrose": "OC[C@H]1O[C@@](CO)(O[C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)[C@@H](O)[C@@H]1O",
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
}

featurizer = NPClassifierFeaturizer(radius=2, n_jobs=-1)
smiles_list = list(MOLECULES.values())
names_list = list(MOLECULES.keys())
features = featurizer.transform(smiles_list)

results = ensemble.predict_from_features(features, smiles_list, check_glycoside=True)

rows = []
for name, res in zip(names_list, results):
    rows.append(
        {
            "molecule": name,
            "pathway": ", ".join(res["pathway"]) or "—",
            "superclass": ", ".join(res["superclass"]) or "—",
            "class": ", ".join(res["class"]) or "—",
            "glycoside": res["isglycoside"],
        }
    )

pd.DataFrame(rows).set_index("molecule")

## 5. Inspect the ontology hierarchy

The ensemble stores precomputed hierarchy mappings built from `index_v1.json`.

In [ ]:
# Which pathways does a class belong to?
cls_name = "Flavonoids"
if cls_name in ensemble._class_labels:
    cls_idx = ensemble._class_labels.index(cls_name)
    paths = ensemble._class_to_pathways.get(cls_idx, [])
    supers = ensemble._class_to_superclasses.get(cls_idx, [])
    print(f"Class '{cls_name}'")
    print(f"  → pathways    : {[ensemble._pathway_labels[i] for i in paths]}")
    print(f"  → superclasses: {[ensemble._superclass_labels[i] for i in supers]}")
else:
    print(f"'{cls_name}' not in class labels — check spelling.")

## 6. Tune voting thresholds

Lower thresholds → more (possibly noisy) predictions; higher → fewer but more confident.

In [ ]:
smi = MOLECULES["quercetin"]

for pw_thr, sup_thr, cls_thr in [
    (0.5, 0.3, 0.1),
    (0.3, 0.2, 0.05),
    (0.7, 0.5, 0.2),
]:
    ensemble.pathway_threshold = pw_thr
    ensemble.superclass_threshold = sup_thr
    ensemble.class_threshold = cls_thr
    r = ensemble.predict(smi)
    print(f"pw={pw_thr}  sup={sup_thr}  cls={cls_thr}")
    print(f"  pathway={r['pathway']}  superclass={r['superclass']}  class={r['class']}")

# Reset to defaults
ensemble.pathway_threshold = 0.5
ensemble.superclass_threshold = 0.3
ensemble.class_threshold = 0.1

## 7. Build ensemble directly from checkpoint paths

Convenience constructor that loads models and assembles the ensemble in one call.

In [ ]:
ensemble2 = NPClassifierEnsemble.from_checkpoints(
    pathway_ckpt=PATHWAY_CKPT,
    superclass_ckpt=SUPERCLASS_CKPT,
    class_ckpt=CLASS_CKPT,
)

r2 = ensemble2.predict("O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12")
print("Quercetin:", r2)